In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, size, lit
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Tokenizer, HashingTF, IDF
from pyspark.ml import Pipeline
from pyspark.ml.feature import PCA





In [3]:
# Create a Spark session
spark = SparkSession.builder.appName("Milestone2").getOrCreate()



In [4]:
# Load JSONL files
df = spark.read.json("gs://imrainprojectbucket/notebooks/jupyter/Data/meta_*.jsonl")

In [5]:
# Print all column names
print(df.columns)


# Selecting a few columns to display
df.select("main_category", "title", "average_rating", "rating_number", "price", "author", "bought_together").show(10, truncate=True)


['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
+-------------+--------------------+--------------+-------------+-----+------+---------------+
|main_category|               title|average_rating|rating_number|price|author|bought_together|
+-------------+--------------------+--------------+-------------+-----+------+---------------+
|   All Beauty|Howard LC0008 Lea...|           4.8|           10| null|  null|           null|
|   All Beauty|Yes to Tomatoes D...|           4.5|            3| null|  null|           null|
|   All Beauty|Eye Patch Black A...|           4.4|           26| null|  null|           null|
|   All Beauty|Tattoo Eyebrow St...|           3.1|          102| null|  null|           null|
|   All Beauty|Precision Plunger...|           4.3|            7| null|  null|           null|
|   All Beauty|Lurrose 100Pcs Fu

In [6]:
# Count total rows before dropping nulls
# total_rows_before = df.count()
# print(f"Total rows before dropping nulls in 'price': {total_rows_before}") # 5752442 rows

# Drop the 'images' and 'videos' columns
df_dropped = df.drop("images", "videos", "bought_together")

# Drop rows where the 'price' column is null
df_prefiltered = df_dropped.filter(df_dropped["price"].isNotNull())
# df_filtered = df.na.drop()




# Define the columns to check for "emptiness"
columns_to_check = ['main_category', 'title', 'categories', 'features', 'store']

# Construct a filter condition
# Adjust the conditions based on your data structure and requirements.
condition = (
#     (col('main_category') != lit('')) & (col('main_category').isNotNull()) &
    (col('title') != lit('')) & (col('title').isNotNull()) &
    (size('categories') > 0) &
    (size('features') > 0) &
    (col('store') != lit('')) & (col('store').isNotNull())
)

# Filter the DataFrame based on the constructed condition
df_filtered = df_prefiltered.filter(condition)


# Count total rows after dropping nulls
total_rows_after = df_filtered.count()
print(f"Total rows after dropping nulls in 'price': {total_rows_after}")


Total rows after dropping nulls in 'price': 1031311


In [7]:
# Selecting a few columns to display
df_filtered.select("main_category", "title", "average_rating", "rating_number", "price", "categories", "features", "store").show(10, truncate=True)


+--------------------+--------------------+--------------+-------------+-----+--------------------+----------------------+---------+
|       main_category|               title|average_rating|rating_number|price|          categories|              features|    store|
+--------------------+--------------------+--------------+-------------+-----+--------------------+----------------------+---------+
|Tools & Home Impr...|Whirlpool W109185...|           3.8|           12|25.07|[Appliances, Part...|  [This is a Genuin...|Whirlpool|
|          Appliances|WD12X10327 Rack R...|           4.6|          323| 8.99|[Appliances, Part...|[【PARTS NUMBER】Th...|AMI PARTS|
|Tools & Home Impr...|SAMSUNG Cap-Handl...|           4.0|           12|53.68|[Appliances, Part...|  [Fits Samsung Ref...|  SAMSUNG|
|Tools & Home Impr...|WP67003405 670034...|           4.1|            4|16.52|[Appliances, Part...|  [WP67003405 Pivot...|PartsBroz|
|         Amazon Home|Chenhon Fridge Ha...|           4.2|           43

In [8]:
from pyspark.sql.functions import concat_ws
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Ensure this step is applied correctly before defining the pipeline stages
df_filtered = df_filtered.withColumn("categories_str", concat_ws(", ", "categories"))
# df_filtered.cache()

indexers = [
    StringIndexer(inputCol=column, outputCol=column + "_Index", handleInvalid="keep")
    for column in ['main_category', 'store', 'categories_str'] 
]

# Define OneHotEncoder for the indexed columns
encoders = [
    OneHotEncoder(inputCol=indexer.getOutputCol(), outputCol=indexer.getInputCol() + "_Vec")
    for indexer in indexers
]

# Pipeline stages
stages = indexers + encoders

# Define the pipeline
pipeline = Pipeline(stages=stages)

# Fit and Transform
model = pipeline.fit(df_filtered)
df_transformed = model.transform(df_filtered)


In [9]:

df_transformed.select(
    "main_category", "main_category_Vec",
    "store", "store_Vec",
    "categories_str", "categories_str_Vec",  # Updated this line
    "title", "average_rating", "rating_number", "price", "features"
).show(10, truncate=True)


+--------------------+-----------------+---------+--------------------+--------------------+------------------+--------------------+--------------+-------------+-----+----------------------+
|       main_category|main_category_Vec|    store|           store_Vec|      categories_str|categories_str_Vec|               title|average_rating|rating_number|price|              features|
+--------------------+-----------------+---------+--------------------+--------------------+------------------+--------------------+--------------+-------------+-----+----------------------+
|Tools & Home Impr...|   (38,[3],[1.0])|Whirlpool| (140457,[10],[1.0])|Appliances, Parts...|  (7315,[9],[1.0])|Whirlpool W109185...|           3.8|           12|25.07|  [This is a Genuin...|
|          Appliances|  (38,[13],[1.0])|AMI PARTS|(140457,[330],[1.0])|Appliances, Parts...|(7315,[386],[1.0])|WD12X10327 Rack R...|           4.6|          323| 8.99|[【PARTS NUMBER】Th...|
|Tools & Home Impr...|   (38,[3],[1.0])|  SAMSU

In [10]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

# Tokenize text
tokenizer = Tokenizer(inputCol="title", outputCol="title_words")
df_tokenized = tokenizer.transform(df_transformed)

# Apply HashingTF and then IDF to get TF-IDF features
hashingTF = HashingTF(inputCol="title_words", outputCol="rawFeatures", numFeatures=20)
featurizedData = hashingTF.transform(df_tokenized)
idf = IDF(inputCol="rawFeatures", outputCol="title_features")
idfModel = idf.fit(featurizedData)
df_tfidf = idfModel.transform(featurizedData)


df_transformed = df_tfidf.withColumn("features_str", concat_ws(" ", "features"))
tokenizer_features = Tokenizer(inputCol="features_str", outputCol="features_words")
df_tokenized_features = tokenizer_features.transform(df_transformed)

# Apply HashingTF to the tokenized 'features' text
hashingTF_features = HashingTF(inputCol="features_words", outputCol="rawFeatures_features", numFeatures=20)
featurizedData_features = hashingTF_features.transform(df_tokenized_features)

# Apply IDF to the hashed 'features'
idf_features = IDF(inputCol="rawFeatures_features", outputCol="features_features")
idfModel_features = idf_features.fit(featurizedData_features)
df_tfidf = idfModel_features.transform(featurizedData_features)


# The final project used hashing on Store for the Generalized Linear Regression model.
# Keep the original one-hot encoded Store feature for the other models, and also
# create a hashed Store feature for GLR.
tokenizer_store = Tokenizer(inputCol="store", outputCol="store_words")
df_store_tokenized = tokenizer_store.transform(df_tfidf)

hashingTF_store = HashingTF(
    inputCol="store_words",
    outputCol="store_hash",
    numFeatures=20
)

df_tfidf = hashingTF_store.transform(df_store_tokenized)


In [11]:
df_tfidf.select(
    "main_category_Vec",
    "store_Vec",
    "store_hash",
    "categories_str_Vec",
    "average_rating", "rating_number", "price", "title_features", "features_features"
).show(10, truncate=True)


+-----------------+--------------------+------------------+--------------+-------------+-----+--------------------+--------------------+
|main_category_Vec|           store_Vec|categories_str_Vec|average_rating|rating_number|price|      title_features|   features_features|
+-----------------+--------------------+------------------+--------------+-------------+-----+--------------------+--------------------+
|   (38,[3],[1.0])| (140457,[10],[1.0])|  (7315,[9],[1.0])|           3.8|           12|25.07|(20,[7,10,14],[0....|(20,[0,1,10,11,12...|
|  (38,[13],[1.0])|(140457,[330],[1.0])|(7315,[386],[1.0])|           4.6|          323| 8.99|(20,[1,2,3,4,5,6,...|(20,[0,1,2,3,4,5,...|
|   (38,[3],[1.0])| (140457,[33],[1.0])|(7315,[293],[1.0])|           4.0|           12|53.68|(20,[2,5,14,19],[...|(20,[0,2,5,11,12,...|
|   (38,[3],[1.0])|(140457,[247],[1.0])|  (7315,[9],[1.0])|           4.1|            4|16.52|(20,[2,3,4,5,6,7,...|(20,[0,1,2,3,4,5,...|
|   (38,[5],[1.0])|(140457,[75696],[...| 

In [11]:
# Assuming df_tfidf already includes "title_features" and "features_features"
df_final = df_tfidf.select(
    "main_category_Vec",
    "store_Vec",
    "store_hash",
    "categories_str_Vec",
    "average_rating", "rating_number", "price", "title_features", "features_features"
)

df_final = df_final.withColumn("price", col("price").cast("float"))
df_final = df_final.withColumn("rating_number", col("rating_number").cast("float"))
df_final = df_final.withColumn("average_rating", col("average_rating").cast("float"))

# Cache df_final for quick access
# df_final.cache()

# To verify caching and view the DataFrame
df_final.show()


+-----------------+--------------------+------------------+--------------+-------------+-----+--------------------+--------------------+
|main_category_Vec|           store_Vec|categories_str_Vec|average_rating|rating_number|price|      title_features|   features_features|
+-----------------+--------------------+------------------+--------------+-------------+-----+--------------------+--------------------+
|   (38,[3],[1.0])| (140457,[10],[1.0])|  (7315,[9],[1.0])|           3.8|         12.0|25.07|(20,[7,10,14],[0....|(20,[0,1,10,11,12...|
|  (38,[13],[1.0])|(140457,[330],[1.0])|(7315,[386],[1.0])|           4.6|        323.0| 8.99|(20,[1,2,3,4,5,6,...|(20,[0,1,2,3,4,5,...|
|   (38,[3],[1.0])| (140457,[33],[1.0])|(7315,[293],[1.0])|           4.0|         12.0|53.68|(20,[2,5,14,19],[...|(20,[0,2,5,11,12,...|
|   (38,[3],[1.0])|(140457,[247],[1.0])|  (7315,[9],[1.0])|           4.1|          4.0|16.52|(20,[2,3,4,5,6,7,...|(20,[0,1,2,3,4,5,...|
|   (38,[5],[1.0])|(140457,[75696],[...| 

In [15]:
import gc

del df
del df_transformed
del df_tfidf
del df_filtered
del df_dropped
del df_prefiltered

# Manually trigger garbage collection
gc.collect()


263

In [4]:
# Saving intermediate data helped avoid rerunning the expensive preprocessing steps
# when Google Cloud disconnected during the project.

checkpoint_path = "gs://imrainprojectbucket/notebooks/jupyter/Data/df_final_parquet_final"

# Run once after rebuilding the final feature set:
# df_final.write.parquet(checkpoint_path, mode="overwrite")

# To resume later without rerunning preprocessing, uncomment:
# df_final = spark.read.parquet(checkpoint_path)


# Q2

For the features used in the regression model, we wll use the following:
"main_category", "title", "average_rating", "rating_number", "price", "categories", "features", "store"

This is because these are the main features that have to do with information on the product. other features like images and videos are incompatible, while others still like ansin or bought_together dont have to do with the product or are completely empty. 

We didnt do PCA or any feature reducing algorithms due to the fact that we already reduced the data enough to be managable and to essential features.

# Modeling

In [5]:
# Save the DataFrame as CSV
# df_final.write.csv("gs://bigdataprojectbucketisa/notebooks/jupyter/Data/df_final.csv", mode="overwrite", header=True)
# df_final = spark.read.csv("gs://bigdataprojectbucketisa/notebooks/jupyter/Data/df_final.csv", header=True, inferSchema=True)

df_final.printSchema()

root
 |-- main_category_Vec: vector (nullable = true)
 |-- store_Vec: vector (nullable = true)
 |-- categories_str_Vec: vector (nullable = true)
 |-- average_rating: float (nullable = true)
 |-- rating_number: float (nullable = true)
 |-- price: float (nullable = true)
 |-- title_features: vector (nullable = true)
 |-- features_features: vector (nullable = true)



In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Assemble features into a single vector
featureCols = ["main_category_Vec", "store_Vec", "categories_str_Vec", "average_rating", "rating_number", "title_features", "features_features"]
assembler = VectorAssembler(inputCols=featureCols, outputCol="features")

# Add the assembler to the stages of your pipeline
stages = [assembler]


In [7]:
# Split the data
(trainData, testData) = df_final.randomSplit([0.7, 0.3], seed=42)
trainData.cache()
testData.cache()


# Handling null values - Option 1: Drop any rows with null values
trainData = trainData.dropna()

In [ ]:

# Linear Regression Model
lr = LinearRegression(featuresCol="features", labelCol="price")
stages += [lr]

# Create a Pipeline
pipeline = Pipeline(stages=stages)

# Train model
model = pipeline.fit(trainData)


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# Make predictions
predictionsTrain = model.transform(trainData)

# Select example rows to display
predictionsTrain.select("prediction", "price", "features").show(5)

# Evaluate the model
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmseTrain = evaluator.evaluate(predictionsTrain)
print("Root Mean Squared Error (RMSE) on training data = %g" % rmseTrain)


+------------------+-----+--------------------+
|        prediction|price|            features|
+------------------+-----+--------------------+
|16.593899288694782|15.99|(147852,[38,14050...|
| 31.49151305811401|18.99|(147852,[38,14055...|
|33.456012307338106|29.95|(147852,[38,14058...|
|  49.3317337453566|28.99|(147852,[38,14064...|
|28.951851730636175|13.99|(147852,[38,14064...|
+------------------+-----+--------------------+
only showing top 5 rows

Root Mean Squared Error (RMSE) on training data = 110.746


In [10]:
# Drop rows with null values in the test data
testData = testData.dropna(subset=["price"])


# Evaluate Linear Regression on test data
predictionsTestLR = model.transform(testData)
rmseTestLR = evaluator.evaluate(predictionsTestLR)
print("RMSE on test data (LR) = %g" % rmseTestLR)


RMSE on test data (LR) = 293.549


The RMSE of the train data was 110 while the test error was 293. currently google cloud is having issues connected and running code, so i cannot get the output messeges

In [11]:
# Random Forest
from pyspark.ml.regression import RandomForestRegressor

# Define the Random Forest model
rf = RandomForestRegressor(featuresCol="features", labelCol="price")

# Reuse the assembler from previous steps and create the stages array
stages = [assembler, rf]  # Use the same feature assembler as before


In [ ]:
# Create a Pipeline
pipelineRF = Pipeline(stages=stages)

# Train the Random Forest model
modelRF = pipelineRF.fit(trainData)


In [ ]:
# Make predictions
predictionsTrainRF = modelRF.transform(trainData)

# Evaluate the model
rmseTrainRF = evaluator.evaluate(predictionsTrainRF)
print("Root Mean Squared Error (RMSE) on training data (RF) = %g" % rmseTrainRF)


# Evaluate the Random Forest model on test data
predictionsTestRF = modelRF.transform(testData)
rmseTestRF = evaluator.evaluate(predictionsTestRF)
print("RMSE on test data (RF) = %g" % rmseTestRF)


The RMSE of the train data was at 230 while the test error was 180. Currently google cloud is having issues connected and running code, so I cannot get the output messeges

The linear regression model and random forest are able to learn, but dont have the best error. To improve this, we might need to normalize the data or add some regularization into the pipeline for future milestones.

For the next milestone, we need to also save some of the dataframes so we can read it in and start the notebook halfway through. Google cloud is having massive issues with trying to run code and remaining connected while the code is running, dispite my internet connect working fine. The kernel will start at "busy" state when you execute a code block, then simply switch to "connected" and get stuck there while the code block is running and the code never finishes. This effectively manifests as a timer for how long google cloud will be functional.

NOTE: After we recorded the video, the linear regression finished with updated errors, so the errors presented in the video may not be up to date.

# Final Milestone Additions

- normalization inside the model pipelines
- Gradient Boosted Tree Regression
- Generalized Linear Regression
- hashing the Store feature for the GLR model
- TPE / Hyperopt tuning of the Linear Regression model



In [ ]:
from pyspark.ml.feature import Normalizer
from pyspark.ml.regression import GBTRegressor, GeneralizedLinearRegression

# Final feature set used by the models that used normalization
featureColsFinal = [
    "main_category_Vec",
    "store_Vec",
    "categories_str_Vec",
    "average_rating",
    "rating_number",
    "title_features",
    "features_features"
]

# Assemble first, then normalize inside the model pipeline
assemblerFinal = VectorAssembler(
    inputCols=featureColsFinal,
    outputCol="assembled_features"
)

normalizer = Normalizer(
    inputCol="assembled_features",
    outputCol="features",
    p=2.0
)


## Linear Regression with Normalization

For the final milestone we added normalization to the prediction pipelines. Linear Regression remained the baseline model.


In [ ]:
# Final Linear Regression pipeline
lrFinal = LinearRegression(
    featuresCol="features",
    labelCol="price"
)

pipelineLRFinal = Pipeline(stages=[
    assemblerFinal,
    normalizer,
    lrFinal
])

modelLRFinal = pipelineLRFinal.fit(trainData)

# Evaluate on training data
predictionsTrainLRFinal = modelLRFinal.transform(trainData)
rmseTrainLRFinal = evaluator.evaluate(predictionsTrainLRFinal)
print("Root Mean Squared Error (RMSE) on training data (Final LR) = %g" % rmseTrainLRFinal)

# Evaluate on test data
predictionsTestLRFinal = modelLRFinal.transform(testData)
rmseTestLRFinal = evaluator.evaluate(predictionsTestLRFinal)
print("RMSE on test data (Final LR) = %g" % rmseTestLRFinal)


## Random Forest with Normalization


In [ ]:
# Final Random Forest model
rfFinal = RandomForestRegressor(
    featuresCol="features",
    labelCol="price"
)

pipelineRFFinal = Pipeline(stages=[
    assemblerFinal,
    normalizer,
    rfFinal
])

modelRFFinal = pipelineRFFinal.fit(trainData)

# Evaluate on training data
predictionsTrainRFFinal = modelRFFinal.transform(trainData)
rmseTrainRFFinal = evaluator.evaluate(predictionsTrainRFFinal)
print("Root Mean Squared Error (RMSE) on training data (Final RF) = %g" % rmseTrainRFFinal)

# Evaluate on test data
predictionsTestRFFinal = modelRFFinal.transform(testData)
rmseTestRFFinal = evaluator.evaluate(predictionsTestRFFinal)
print("RMSE on test data (Final RF) = %g" % rmseTestRFFinal)


## Gradient Boosted Tree Regression

Gradient Boosted Tree Regression was one of the additional models tested in the final project.


In [ ]:
# Gradient Boosted Tree Regression
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="price"
)

pipelineGBT = Pipeline(stages=[
    assemblerFinal,
    normalizer,
    gbt
])

modelGBT = pipelineGBT.fit(trainData)

# Evaluate on training data
predictionsTrainGBT = modelGBT.transform(trainData)
rmseTrainGBT = evaluator.evaluate(predictionsTrainGBT)
print("Root Mean Squared Error (RMSE) on training data (GBT) = %g" % rmseTrainGBT)

# Evaluate on test data
predictionsTestGBT = modelGBT.transform(testData)
rmseTestGBT = evaluator.evaluate(predictionsTestGBT)
print("RMSE on test data (GBT) = %g" % rmseTestGBT)


## Generalized Linear Regression

For Generalized Linear Regression, the final project used hashing on the Store column instead of the one-hot encoded Store feature. The report also states that normalization was not used for this model.


In [ ]:
# Feature set for Generalized Linear Regression
# Store uses the hashed feature created during preprocessing.
featureColsGLR = [
    "main_category_Vec",
    "store_hash",
    "categories_str_Vec",
    "average_rating",
    "rating_number",
    "title_features",
    "features_features"
]

assemblerGLR = VectorAssembler(
    inputCols=featureColsGLR,
    outputCol="features"
)

glr = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="price",
    family="gaussian",
    link="identity"
)

pipelineGLR = Pipeline(stages=[
    assemblerGLR,
    glr
])

modelGLR = pipelineGLR.fit(trainData)

# Evaluate on training data
predictionsTrainGLR = modelGLR.transform(trainData)
rmseTrainGLR = evaluator.evaluate(predictionsTrainGLR)
print("Root Mean Squared Error (RMSE) on training data (GLR) = %g" % rmseTrainGLR)

# Evaluate on test data
predictionsTestGLR = modelGLR.transform(testData)
rmseTestGLR = evaluator.evaluate(predictionsTestGLR)
print("RMSE on test data (GLR) = %g" % rmseTestGLR)


## TPE Hyperparameter Optimization

Grid Search and Random Search were too computationally expensive for the project, so the final milestone used Tree-structured Parzen Estimator (TPE) optimization with Hyperopt. The parameters tuned for Linear Regression were `regParam` and `elasticNetParam`.


In [ ]:
# Hyperopt may need to be installed in the notebook environment:
# pip install hyperopt

import math
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# Use a validation split so the test set stays separate from hyperparameter tuning
(trainTune, validationData) = trainData.randomSplit([0.8, 0.2], seed=42)

search_space = {
    "regParam": hp.loguniform("regParam", math.log(1e-4), math.log(1.0)),
    "elasticNetParam": hp.uniform("elasticNetParam", 0.0, 1.0)
}

def objective(params):
    lr_tune = LinearRegression(
        featuresCol="features",
        labelCol="price",
        regParam=float(params["regParam"]),
        elasticNetParam=float(params["elasticNetParam"])
    )

    pipeline_tune = Pipeline(stages=[
        assemblerFinal,
        normalizer,
        lr_tune
    ])

    model_tune = pipeline_tune.fit(trainTune)
    predictions_validation = model_tune.transform(validationData)
    rmse_validation = evaluator.evaluate(predictions_validation)

    return {
        "loss": rmse_validation,
        "status": STATUS_OK
    }

trials = Trials()

best_params = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials
)

print("Best parameters found by TPE:")
print(best_params)


In [ ]:
# Train Linear Regression again using the best TPE parameters
lrOptimized = LinearRegression(
    featuresCol="features",
    labelCol="price",
    regParam=float(best_params["regParam"]),
    elasticNetParam=float(best_params["elasticNetParam"])
)

pipelineOptimized = Pipeline(stages=[
    assemblerFinal,
    normalizer,
    lrOptimized
])

modelOptimized = pipelineOptimized.fit(trainData)

predictionsTrainOptimized = modelOptimized.transform(trainData)
predictionsTestOptimized = modelOptimized.transform(testData)

rmseTrainOptimized = evaluator.evaluate(predictionsTrainOptimized)
rmseTestOptimized = evaluator.evaluate(predictionsTestOptimized)

print("Root Mean Squared Error (RMSE) on training data (Tuned LR) = %g" % rmseTrainOptimized)
print("RMSE on test data (Tuned LR) = %g" % rmseTestOptimized)


# Final Results

The final project report recorded the following results:

| Model | Train RMSE | Test RMSE |
| --- | ---: | ---: |
| Linear Regression | 113.961 | 115.333 |
| Random Forest | 114.817 | 135.849 |
| Gradient Boosted Tree Regression | 139.123 | 140.614 |
| Generalized Linear Regression | 128.699 | 134.895 |

Linear Regression had the lowest test RMSE and was also the fastest model in the final project. TPE hyperparameter optimization did not significantly improve the model.

The final report also noted that the errors were still too high for a deployable price prediction system. Limited compute meant that only a portion of the original Amazon dataset could be used, and Google Cloud / Dataproc runtime issues made the larger models difficult to run.
